# Heart Rate Time Series Forecasting

Predicting the next 20 minutes of Lifetouch Heart Rate from ~4 hours of sensor data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.holtwinters import Holt
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')
plt.rcParams['figure.facecolor'] = 'white'

## 1. Load Data

In [ ]:
df = pd.read_csv('data/PT_Train.csv')
df['Timestamp (GMT)'] = pd.to_datetime(df['Timestamp (GMT)'], format='%d/%m/%Y %H:%M')
df.set_index('Timestamp (GMT)', inplace=True)
df.sort_index(inplace=True)
print(f"Rows: {len(df)}")

## 2. Outlier Cleaning
Sensor errors (~61,441 bpm) are replaced using a **rolling median** (window=5) for robustness against spikes.

In [ ]:
hr = df['Lifetouch Heart Rate'].copy()
hr[hr > 250] = np.nan
hr = hr.fillna(hr.rolling(5, min_periods=1).median()).ffill().bfill()
print(f"Range: {hr.min():.0f} - {hr.max():.0f} bpm")

plt.figure(figsize=(11, 3.5))
plt.plot(hr, color='#16a085', linewidth=1.5)
plt.title('Heart Rate (Rolling Median Imputation)')
plt.ylabel('BPM')
plt.xlabel('Time')
plt.tight_layout()
plt.savefig('ts_history_v3.png', dpi=150)
plt.show()

## 3. Stationarity

In [ ]:
adf = adfuller(hr.dropna())
print(f"ADF p-value: {adf[1]:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 3))
plot_acf(hr.dropna(), ax=ax[0], lags=30)
plot_pacf(hr.dropna(), ax=ax[1], lags=30)
plt.tight_layout()
plt.show()

## 4. Train / Validation
Hold out last **25** observations.

In [ ]:
n_val = 25
train, val = hr.iloc[:-n_val], hr.iloc[-n_val:]
print(f"Train: {len(train)} | Val: {len(val)}")

## 5. Models
- **ARIMA(0,1,0)** (Random Walk)
- **Holt's Linear** (Double Exponential Smoothing)
- **ARIMA(2,1,0)**

In [ ]:
def eval_model(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{name}: MAE={mae:.3f} RMSE={rmse:.3f}")
    return mae, rmse

# Random Walk
rw = ARIMA(train, order=(0, 1, 0)).fit()
rw_fc = np.array(rw.forecast(steps=n_val))
rw_mae, rw_rmse = eval_model(val.values, rw_fc, "ARIMA(0,1,0) Random Walk")

# Holt
holt = Holt(train, initialization_method='estimated').fit()
holt_fc = np.array(holt.forecast(n_val))
holt_mae, holt_rmse = eval_model(val.values, holt_fc, "Holt Linear")

# ARIMA(2,1,0)
ar2 = ARIMA(train, order=(2, 1, 0)).fit()
ar2_fc = np.array(ar2.forecast(steps=n_val))
ar2_mae, ar2_rmse = eval_model(val.values, ar2_fc, "ARIMA(2,1,0)")

In [ ]:
models = {'ARIMA(0,1,0)': (rw_mae, rw_rmse), 'Holt': (holt_mae, holt_rmse), 'ARIMA(2,1,0)': (ar2_mae, ar2_rmse)}
best = min(models, key=lambda k: models[k][0])
print(f"\nBest (MAE): {best}")

## 6. Forecast Plot

In [ ]:
plt.figure(figsize=(11, 3.5))
plt.plot(train.index[-35:], train.iloc[-35:], label='Train', color='gray')
plt.plot(val.index, val, label='Actual', color='black', linewidth=2)
plt.plot(val.index, rw_fc, '--', label='Random Walk')
plt.plot(val.index, holt_fc, '--', label='Holt')
plt.plot(val.index, ar2_fc, '--', label='ARIMA(2,1,0)')
plt.legend(fontsize=8)
plt.title('Validation Forecasts')
plt.tight_layout()
plt.savefig('ts_forecast_v3.png', dpi=150)
plt.show()

## 7. Final 20-Step Forecast

In [ ]:
print(f"Retraining {best} on full series...")
if best == 'ARIMA(0,1,0)':
    final = ARIMA(hr, order=(0, 1, 0)).fit()
    fc = final.forecast(steps=20)
elif best == 'Holt':
    final = Holt(hr, initialization_method='estimated').fit()
    fc = final.forecast(20)
else:
    final = ARIMA(hr, order=(2, 1, 0)).fit()
    fc = final.forecast(steps=20)

out = pd.DataFrame({'Observation (Future Minute)': range(1, 21), 'Predicted_Heart_Rate': np.round(fc.values, 2)})
out.to_csv('test-predictions_v3.csv', index=False)
print("Done.")
out